In [1]:
import multiprocessing
import pathlib

import polars
from gensim.models import word2vec
from sklearn import model_selection as sk_model_selection
from sklearn import naive_bayes as sk_nb
from sklearn import preprocessing as sk_pp

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_DIR / "output_dataset"
VECTOR_SIZE = 32
PROTOCOLS = ["coap", "coaps", "oscore"]
LINK_LAYER = [""]
NETWORK_SETUP = ["d1", "d2", "p1", "p2"]
DATA_FORMAT = ["json", "cbor"]
DNS_FORMAT = ["dns_message", "dns_cbor"]

LINK_LAYER_READABLE = {
    "": "eth",
    "-schc": "schc",
}
DNS_FORMAT_READABLE = {
    "dns_message": "application/dns-message",
    "dns_cbor": "application/dns+cbor",
}

# Preparations

In [2]:
SCENARIO = "oscore-p2_json_dns_message"

In [3]:
df = polars.read_csv(INPUT_PATH / f"{SCENARIO}.training.csv", separator=";")

In [4]:
nibbles, byte_size = df["eth.payload"].str.len_chars().max(), 2

df = df.with_columns(
    polars.col("eth.payload").map_elements(
        lambda msg: [ msg[i:i+byte_size] for i in range(0, nibbles, byte_size) ],
        return_dtype=list[str],
    ).alias("eth.payload list")
)

In [ ]:
VECTOR_SIZE = 32

model = word2vec.Word2Vec(
    df["eth.payload list"].to_list(),
    workers=multiprocessing.cpu_count(),
    vector_size=VECTOR_SIZE,
    min_count=1,
    window=3,
    sg=0
)

In [ ]:
df_dns = df.filter(
    polars.col("client.type") == "dns"
)[["eth.payload list"]].with_columns(
    vectors=polars.col("eth.payload list").map_elements(
        lambda words: [model.wv[word] for word in words],
        return_dtype=polars.datatypes.List(
            polars.datatypes.Array(polars.Float32, VECTOR_SIZE)
        ),
    ),
    label=1,
)[["vectors", "label"]]
df_dns.write_parquet(INPUT_PATH / f"{SCENARIO}.model.dns.parquet", compression="zstd", compression_level=10)

In [ ]:
df_dns = polars.read_parquet(INPUT_PATH / f"{SCENARIO}.model.dns.parquet")
df_dns

In [ ]:
x = merged_df_we.iloc[:, :-1]
y = merged_df_we.iloc[:, -1:]

x = x.to_numpy()
y = y.to_numpy()

In [ ]:
x_train, x_test, y_train, y_test = sk_model_selection.train_test_split(x, y, test_size=0.2)

# Naïve Bayes

In [ ]:
scaler = sk_pp.MinMaxScaler()
x_train_minmax = scaler.fit_transform(x_train)
x_test_minmax = scaler.transform(x_test)

classifier = sk_nb.MultinomialNB().fit(x_train_minmax, y_train)

In [ ]:
y_test_predictions = classifier.predict(x_test_minmax)
conf_matrix = confusion_matrix(y_test, y_test_predictions)
accuracy = accuracy_score(y_test, y_test_predictions)
precision = precision_score(y_test, y_test_predictions)
recall = recall_score(y_test, y_test_predictions)
f1score = f1_score(y_test, y_test_predictions)

print(f"Accuracy = {accuracy:3g}")
print(f"Precision = {precision:3g}")
print(f"Recall = {recall:3g}")
print(f"F1 Score = {f1score:3g}")
conf_matrix